In [1]:
# ######### used this part for fixing problems running on ARC #
import os
os.environ['HF_HOME'] = '/projects/cesca-cv/ishtiaque/models/'
os.environ['HF_HUB_CACHE'] = '/projects/cesca-cv/ishtiaque/models/'
os.environ['XDG_CACHE_HOME'] = '/projects/cesca-cv/ishtiaque/models/'
os.environ['NB_USER'] = 'ishtiahmed'
os.environ['TRANSFORMERS_CACHE'] = '/projects/cesca-cv/ishtiaque/models/'
os.environ['HF_DATASETS_CACHE'] = '/projects/cesca-cv/ishtiaque/models/'

'"\n ssh -f -N -L 8082:fal039:60985 ishtiaqueahmedk@falcon1.arc.vt.edu \n'

In [4]:
import torch
import os 
import json
import random
from tqdm import tqdm
from collections import Counter 

import numpy as np
from openai import OpenAI
import argparse

In [ ]:
# dedicated gpu server through interactive apps
# Modify OpenAI's API key and API base to use the server.
openai_api_key = "xxxxx"
openai_api_base = "http://localhost:8082/v1"


client = OpenAI(
        api_key=openai_api_key,
        base_url=openai_api_base,
    )

models = client.models.list()
model = models.data[0].id
# print(models.data[0].id)
print(model)

Llama-4-Scout-17B-16E-Instruct-Q4_K_M-00001-of-00002.gguf


In [6]:
# for singl_modl in models.data:
#     print(singl_modl.id)

In [4]:
import base64

def encode_base64_content_from_file(file_path: str) -> str:
    """Encode a local file content to base64 format."""

    with open(file_path, "rb") as file:
        file_content = file.read()
        result = base64.b64encode(file_content).decode("utf-8")

    return result

In [5]:



image_file = "/home/ishtiaqueahmedk/Research/VLM/fine_grained/stored/Black_footed_Albatross_var1.jpg"

image_file = "/home/ishtiaqueahmedk/Research/VLM/speed.jpg" #speed.jpg #bf1.jpg
image_file = "/home/ishtiaqueahmedk/Research/VLM/masked/cyclist masked .png" #speed.jpg #bf1.jpg
image_file = "/home/ishtiaqueahmedk/Research/VLM/fine_grained/multi_image_prac/1/bagghhuu.jpg"


## Use base64 encoded local image in the payload
if os.path.exists(image_file):
    local_image_base64 = encode_base64_content_from_file(image_file)
    chat_completion_from_local_image_base64 = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "What is in this image?"},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{local_image_base64}"
                        },
                    },
                ],
            }
        ],
        model=model,
        # max_completion_tokens=max_completion_tokens,
    )

    result = chat_completion_from_local_image_base64.choices[0].message.content
    print("Chat completion output from base64 encoded local image:", result)
else:
    print(f"Local image file not found at {image_file}, skipping local file test.")

Chat completion output from base64 encoded local image: The image depicts a tiger, showcasing its distinctive orange and white fur with black stripes. The tiger's facial features are prominent, including its yellow eyes, white fur around the mouth and chin, and black stripes on its forehead and cheeks. Its ears are perked up, and it appears to be looking directly at the camera.

**Key Features:**

* **Fur:** Orange and white with black stripes
* **Eyes:** Yellow
* **Facial Features:** White fur around the mouth and chin, black stripes on the forehead and cheeks
* **Ears:** Perked up
* **Gaze:** Directly at the camera

**Background:**
The background of the image is blurred, but it appears to be a natural setting with rocks or boulders visible behind the tiger. The overall atmosphere suggests that the tiger is in its natural habitat, possibly in a zoo or wildlife sanctuary.


In [5]:
import re


def get_answer(image_path, query):

    local_image_base64 = encode_base64_content_from_file(image_path)
    chat_completion_from_local_image_base64 = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": query},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{local_image_base64}"
                        },
                    },
                ],
            }
        ],
        model=model,
        # max_completion_tokens=max_completion_tokens,
    )

    result = chat_completion_from_local_image_base64.choices[0].message.content
    # print("Chat completion output from base64 encoded local image:", result)


    match = re.search(r'<answer>(.*?)</answer>', result, re.DOTALL)
    final_answer = match.group(1).strip() if match else result

    # print(result)
    # print(final_answer)
    return final_answer
    


# img_path = "/projects/abbott_lab/Users/ishtiaque/hfmodels/DeepSeek-VL2/images/visual_grounding_1.jpeg"
# new_answ = get_answer(img_path, "What animal is in this image? A.Cat B.Dog C.Giraffe D.Bird E.Tiger. Wrap your final answer in <answer></answer> tags")
# # new_answ = get_answer(img_path, "What model are you?")


# #image_path = "/home/ishtiaqueahmedk/Research/VLM/fine_grained/multi_image_prac/1/bagghhuu.jpg"
# #"/projects/abbott_lab/Users/ishtiaque/hfmodels/DeepSeek-VL2/images/visual_grounding_1.jpeg"
# #/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images/001.Black_footed_Albatross/Black_Footed_Albatross_0001_796111.jpg


# match = re.search(r'<answer>(.*?)</answer>', new_answ, re.DOTALL)
# final_answer = match.group(1).strip() if match else result


# print(f"\n\n new answer is: {new_answ}")

# print(f"\n\n final answer is: {final_answer}")



# begin bird code

In [6]:
def get_bird_images(images_folder):
# Path to the folder containing bird subfolders
    # images_folder = '/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images'
    
    # Dictionary to store bird names and their corresponding image paths
    bird_images = {}
    
    # Iterate over each subfolder in the images folder
    for folder in os.listdir(images_folder):
        bird_name = folder.split(".")[-1]  # Extract bird name from folder name
        folder_path = os.path.join(images_folder, folder)  # Path to the bird's folder
        
        # Initialize an empty list to store image paths for the current bird
        image_paths = []
        
        # Iterate over the image files in the bird's folder
        for image_file in os.listdir(folder_path):
            image_path = os.path.join(folder_path, image_file)  # Full path to the image file
            image_paths.append(image_path)  # Store the image path
        
        # Store the list of image paths in the dictionary under the bird's name
        bird_images[bird_name] = image_paths
    
    # Now bird_images contains a dictionary where the keys are bird names and the values are lists of image paths
    print(len(bird_images))
    return bird_images



In [7]:
def get_json_data(json_file_name):

    # negated_questions, modified_new_cub_class_descriptions_full_fake, #modified_new_cub_class_descriptionsx #modified_mcqs_description_only2, modified_mcqs_description_only

    # For Task 1 type 1: 
    
    with open(json_file_name, "r", encoding='utf-8') as file: 
        json_data = json.load(file)
    print(len(json_data))
    return json_data






In [8]:
def get_medium_hard_data(json_data):
    
# Use this if the json file contain the medium and hard categories
    medium_data = []
    hard_data = []

    data_counter = 0
    use_partial = False#True
    if use_partial:
        print("using limited data for debugging")
    
    for data in json_data:
        if data['difficulty'] == "Medium":
            medium_data.append(data)
        else:
            hard_data.append(data)

        

        data_counter = data_counter + 1
        if (data_counter>50) and use_partial:
            print(f"stopping at data = {data_counter}")
            break
        
    print(len(medium_data))
    print(len(hard_data))
    return medium_data, hard_data
    

In [9]:
def run_eval(data_partition):


    if ("task_1a" in json_file_name): #correct ClassName
        print("task_1a\n")
    
    
    # Counters for distribution
    true_distribution = Counter()
    predicted_distribution = Counter()
    
    results = []
    
    for i, item in tqdm(enumerate(data_partition)): # for easy part json_data, for medium_data, for hard_data 
        mcq_id = item['mcq_id']
        question = item['question']
        options = item['options']
        correct_answer = item['correct_answer']
    
        if mcq_id not in bird_images or not bird_images[mcq_id]:
            print(f"No image for {mcq_id}") 
            continue
    
        image_paths = bird_images[mcq_id][:5]
    
        # Format the prompt
        # formatted_prompt = f"{question}\n"

        if ("task_1a" in json_file_name): #correct ClassName
            # print("task_1a\n")
            formatted_prompt = f"{question} Ignore the descriptions and focus only on the class names.\n"
        elif ("task_1b" in json_file_name): #correct Description
            formatted_prompt = f"{question} Ignore the class names and focus only on the descriptions.\n"
        else:
            print(f"Error in File Name: {json_file_name}")
            print(asd)

        # formatted_prompt = f"{question} Ignore the class names and focus on the descriptions.\n" # 
        for k in ['A', 'B', 'C', 'D']:  # ['D', 'C', 'B', 'A'] for position bias checking ['A', 'B', 'C', 'D']
            formatted_prompt += f"{k}. {options[k]}\n"
    
        # Final prompt
        prompt = f""" Your answer or response must ONLY be a single index ('A', 'B', 'C', 'D'). Do not response with any other text. 
    
        {formatted_prompt}
    
        Answer: ('A', 'B', 'C', 'D')"""
    
        # Run the model
        for image_path in image_paths:
            model_output = get_answer(image_path, prompt)
            # print("Model Output: ", model_output)
    
            # Extract predicted answer (basic string search, can refine)
            predicted_answer = None
            for option in ['A', 'B', 'C', 'D']:
                if f"{option}" in model_output or f"{option}." in model_output:
                    predicted_answer = option
    
            # Update counters
            true_distribution[correct_answer] += 1
            if predicted_answer:
                predicted_distribution[predicted_answer] += 1
            
            results.append({
                'mcq_id': mcq_id,
                'image_path': image_path,
                'prompt': prompt,
                'model_output': model_output,
                'predicted_answer': predicted_answer,
                'correct_answer': correct_answer,
                'is_correct': predicted_answer == correct_answer
            })
    
    print(f"Results for file: {json_file_name}")
    # Accuracy summary 
    correct = sum(r['is_correct'] for r in results if r['predicted_answer'] is not None)
    total = len(results)
    print(f"Accuracy: {correct}/{total} = {correct / total:.2%}") 
    
    # Print distributions
    print("True Option Distribution:", dict(true_distribution))
    print("Predicted Option Distribution:", dict(predicted_distribution)) 

In [10]:
print("begin running code")


begin running code


# Run-2

In [11]:

# json_files_list = ['negated_questions', 'new_cub_class_descriptions','new_cub_with_class_descriptions', 'modified_new_cub_class_descriptions_full_fake', 'modified_new_cub_class_descriptions_partial_fake', 'incorrect_class_correct_description', 'correct_class_incorrect_description']

bird_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images"
food_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/food-101/food-101/images"
aircraft_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/fgvc-aircraft-2013b/data/test"
dogs_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-dogs/images/Images"
car_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-cars/train"

bird_list = [
    "new_cub_class_descriptions_task_1b",
    "new_cub_class_descriptions_task_1a",
]

food_list = [
    "new_food_class_descriptions_task_1a",
    "new_food_class_descriptions_task_1b",
]

aircraft_list = [
    "new_aircraft_class_descriptions_task_1a",
    "new_aircraft_class_descriptions_task_1b",
]

dogs_list = [
    "new_dogs_class_descriptions_task_1a",
    "new_dogs_class_descriptions_task_1b",
]

car_list = [
    "new_car_class_descriptions_task_1b",
]

# json_files_list = food_list  #################### change!!!!!!!!!!!!!!!!!!!!
all_lists = [car_list]
folder_lists = [car_folder]

# all_lists = [food_list, aircraft_list, dogs_list, car_list]
# folder_lists = [food_folder, aircraft_folder, dogs_folder, car_folder]


for json_files_list, images_folder in zip(all_lists, folder_lists):

    bird_images = get_bird_images(images_folder)
    
    for json_file_name in json_files_list:
        
        json_file_name = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/{json_file_name}.json"
        json_data = get_json_data(json_file_name)
    
        json_data[0]
        # json_data[1]
    
        medium_data, hard_data = get_medium_hard_data(json_data)
        
        print("\n----Medium----")
        run_eval(medium_data)
        print("\n----Hard----")
        run_eval(hard_data)
    

196
392
196
196

----Medium----


196it [23:45,  7.27s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_1b.json
Accuracy: 877/980 = 89.49%
True Option Distribution: {'B': 250, 'C': 300, 'D': 250, 'A': 180}
Predicted Option Distribution: {'B': 257, 'C': 305, 'A': 225, 'D': 193}

----Hard----


196it [23:43,  7.26s/it]

Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_1b.json
Accuracy: 475/980 = 48.47%
True Option Distribution: {'A': 245, 'D': 250, 'B': 245, 'C': 240}
Predicted Option Distribution: {'A': 274, 'D': 99, 'B': 369, 'C': 238}


In [15]:

# json_files_list = ['negated_questions', 'new_cub_class_descriptions','new_cub_with_class_descriptions', 'modified_new_cub_class_descriptions_full_fake', 'modified_new_cub_class_descriptions_partial_fake', 'incorrect_class_correct_description', 'correct_class_incorrect_description']

bird_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images"
food_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/food-101/food-101/images"
aircraft_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/fgvc-aircraft-2013b/data/test"
dogs_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-dogs/images/Images"
car_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-cars/train"

bird_list = [
    "new_cub_class_descriptions_task_1b",
    "new_cub_class_descriptions_task_1a",
]

food_list = [
    "new_food_class_descriptions_task_1a",
    "new_food_class_descriptions_task_1b",
]

aircraft_list = [
    "new_aircraft_class_descriptions_task_1a",
    "new_aircraft_class_descriptions_task_1b",
]

dogs_list = [
    "new_dogs_class_descriptions_task_1a",
    "new_dogs_class_descriptions_task_1b",
]

car_list = [
    "new_car_class_descriptions_task_1a",
    "new_car_class_descriptions_task_1b",
]

# json_files_list = food_list  #################### change!!!!!!!!!!!!!!!!!!!!
all_lists = [bird_list, food_list, aircraft_list, dogs_list, car_list]
folder_lists = [bird_folder, food_folder, aircraft_folder, dogs_folder, car_folder]

# all_lists = [food_list, aircraft_list, dogs_list, car_list]
# folder_lists = [food_folder, aircraft_folder, dogs_folder, car_folder]


for json_files_list, images_folder in zip(all_lists, folder_lists):

    bird_images = get_bird_images(images_folder)
    
    for json_file_name in json_files_list:
        
        json_file_name = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/{json_file_name}.json"
        json_data = get_json_data(json_file_name)
    
        json_data[0]
        # json_data[1]
    
        medium_data, hard_data = get_medium_hard_data(json_data)
        
        print("\n----Medium----")
        run_eval(medium_data)
        print("\n----Hard----")
        run_eval(hard_data)
    

200
400
200
200

----Medium----


200it [20:14,  6.07s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_cub_class_descriptions_task_1b.json
Accuracy: 968/1000 = 96.80%
True Option Distribution: {'D': 245, 'C': 220, 'B': 235, 'A': 300}
Predicted Option Distribution: {'D': 229, 'C': 221, 'B': 252, 'A': 298}

----Hard----


200it [19:56,  5.98s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_cub_class_descriptions_task_1b.json
Accuracy: 423/1000 = 42.30%
True Option Distribution: {'D': 275, 'A': 260, 'B': 215, 'C': 250}
Predicted Option Distribution: {'D': 126, 'C': 226, 'A': 221, 'B': 427}
400
200
200

----Medium----
task_1a



...

Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_1a.json
Accuracy: 922/980 = 94.08%
True Option Distribution: {'B': 250, 'C': 300, 'D': 250, 'A': 180}
Predicted Option Distribution: {'B': 250, 'C': 296, 'A': 184, 'D': 250}

----Hard----
task_1a



196it [23:34,  7.22s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_1a.json
Accuracy: 478/980 = 48.78%
True Option Distribution: {'A': 245, 'D': 250, 'B': 245, 'C': 240}
Predicted Option Distribution: {'C': 261, 'A': 190, 'B': 372, 'D': 157}
392
196
196

----Medium----


157it [18:28,  7.06s/it]


APIConnectionError: Connection error.

# Run-2


In [ ]:
print("done")